# Task-Aware Intelligent Human Resource Management Using Reflection-Equivariant Quantum Neural Networks

**Full implementation of the 10-stage pipeline** (PyTorch + scikit-learn + NumPy; the quantum circuits are simulated exactly with a differentiable state-vector simulator written in PyTorch, so no quantum SDK is required).

| Stage | Component | Cell group |
|---|---|---|
| 1 | Multi-source HR data collection & integration | `Stage 1` |
| 2 | KNN imputation → Robust scaling → Target encoding → Quantile transform → temporal features | `Stage 2` |
| 3 | **TERRN** – heterogeneous graph, relation-aware attention, hypergraph, temporal message passing | `Stage 3` |
| 4 | **RE-WFE** – Z₂ reflection-equivariant workforce feature encoder | `Stage 4` |
| 5 | PCA quantum feature compression, angle / amplitude encoding | `Stage 5` |
| 6 | **REQ-WN** – reflection-equivariant variational quantum network with data re-uploading | `Stage 6` |
| 7 | Constrained multi-agent RL task allocation | `Stage 7` |
| 8 | Contextual bandit, online learning, drift detection (Page-Hinkley), adaptive policy | `Stage 8` |
| 9 | Shapley (SHAP-style) attribution, counterfactual / what-if | `Stage 9` |
| 10 | Metrics, baselines, quantum-resource report, ablations, nested stratified CV, temporal hold-out | `Stage 10` |

**Please read before running**
* `FAST = True` (first code cell) gives a CPU run of roughly 15–30 minutes. Set `FAST = False` for the full-length training used for final numbers.
* **Data.** By default the notebook generates a *synthetic* Employee–Task–Performance world (employees, skills, tasks, deadlines, workloads, outcomes, injected missing values and duplicate log lines). To combine it with the **IBM HR Analytics** dataset set `CFG['IBM_PATH']` to the CSV; its employee attributes (role, department, experience, education, training, satisfaction, performance/involvement, attrition) then replace the synthetic employee table, while skills (an **O\*NET**-style role→skill importance profile, see `make_role_profiles`) and tasks are generated. Organisational records can be mapped to the same schema.
* **Ground truth.** Task outcomes come from a latent simulator (`World.outcome`). It is used (i) to create the historical log and (ii) as the environment for the allocation/bandit experiments (Stages 7–8) – the learning stages never read its parameters. With real logs, replace `World.outcome` by a fitted outcome model or do off-policy evaluation.
* **What "reflection-equivariant" means here.** The TERRN pair embedding is `z = [a ; b]` (*capability half*, *task-demand half*). The reflection `R` swaps the two halves (the mirror of the employee↔task axis). In the basis `x₊=(a+b)/√2`, `x₋=(a−b)/√2` it acts as `(x₊, x₋) → (x₊, −x₋)`; RE-WFE uses block-diagonal maps, invariant gates and an equivariant LayerNorm, so `RE-WFE(R z) = R·RE-WFE(z)` exactly (checked numerically). In the quantum circuit the reflection is the mirror of the qubit register (`q ↔ n−1−q`); tying parameters across mirror pairs and using commuting ZZ entanglers makes the circuit exactly equivariant (also checked). The measurement head is an ordinary linear layer, so final predictions are not constrained to be invariant.
* Embeddings are trained on the training period (temporal split 65 / 15 / 20 %); thresholds are tuned on validation; all reported metrics are on the temporal test block.

## Stage 1 — HR Data Collection and Multi-Source Integration
**Processing:** data integration → schema harmonisation → employee–task mapping → missing-value analysis → duplicate removal.  
**Output:** unified Employee–Task–Performance dataset (one row per assignment, with outcomes: completion, delay, workload risk, suitability, imbalance, productivity).

In [ ]:
import os, math, time, copy, random, warnings
from collections import deque
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

FAST = True
def pick(fast, full): return fast if FAST else full

CFG = dict(
    SEED=42, IBM_PATH=None,           # e.g. "WA_Fn-UseC_-HR-Employee-Attrition.csv"
    N_EMP=240, N_TASKS=4000, N_SKILLS=8, HORIZON=90.0, SEQ_LEN=5,
    D=32, N_QUBITS=6, Q_LAYERS=3, ENCODING='angle',
)
rng_global = np.random.default_rng(CFG['SEED'])

DEPTS = ['Engineering', 'Data', 'Operations', 'Product']
ROLES_BY_DEPT = {'Engineering': ['Developer', 'QA Engineer', 'DevOps'], 'Data': ['Data Scientist', 'Analyst'],
                 'Operations': ['Support', 'DevOps'], 'Product': ['Designer', 'Manager', 'Analyst']}
TASK_TYPES = ['feature', 'bugfix', 'analysis', 'deployment', 'support']
TARGETS = ['complete', 'risk', 'delay', 'suit', 'imbalance', 'productivity']   # 5 binary + 1 regression
PAIR_FEATS = ['skill_match', 'perf_score', 'experience', 'workload_ratio', 'complexity',
              'deadline_slack', 'priority', 'est_duration', 'recent_delay_rate']
sigmoid = lambda x: 1 / (1 + np.exp(-x))


EP = dict(graph=pick(20, 80), q=pick(25, 60), rl_eps=pick(4, 20), rl_tasks=pick(250, 600), stream=pick(400, 800), cv=pick(8, 30))
import matplotlib.pyplot as plt
from IPython.display import display
import torch, torch.nn as nn, torch.nn.functional as F
np.random.seed(CFG['SEED']); random.seed(CFG['SEED']); torch.manual_seed(CFG['SEED'])

In [ ]:
def make_role_profiles(roles, K, rng):
    """Role -> skill-importance profile. Replace with O*NET importance scores if available."""
    prof = {}
    for r in sorted(set(roles)):
        p = rng.beta(1.5, 4, size=K); sig = rng.choice(K, 3, replace=False)
        p[sig] = rng.uniform(0.6, 0.95, 3); prof[r] = p
    return prof

def build_employees(cfg, rng):
    K = cfg['N_SKILLS']; n = cfg['N_EMP']
    if cfg['IBM_PATH'] and os.path.exists(cfg['IBM_PATH']):          # IBM HR Analytics mapping
        raw = pd.read_csv(cfg['IBM_PATH']).sample(n=min(n, 1470), random_state=cfg['SEED']).reset_index(drop=True)
        n = len(raw)
        emp = pd.DataFrame({'role': raw['JobRole'], 'dept': raw['Department'],
            'experience': raw['TotalWorkingYears'].astype(float), 'qualification': raw['Education'].astype(float),
            'training_hours': raw['TrainingTimesLastYear'] * 8.0, 'satisfaction': raw['JobSatisfaction'].astype(float),
            'perf_score': 0.5 * (raw['PerformanceRating'] - 1) / 3 + 0.5 * (raw['JobInvolvement'] - 1) / 3,
            'turnover': (raw['Attrition'] == 'Yes').astype(int)})
    else:
        dept = rng.choice(DEPTS, n, p=[.35, .25, .25, .15])
        role = [rng.choice(ROLES_BY_DEPT[d]) for d in dept]
        exp_ = np.clip(rng.gamma(2.5, 3.0, n), 0.3, 30); qual = rng.integers(1, 6, n).astype(float)
        emp = pd.DataFrame({'role': role, 'dept': dept, 'experience': exp_, 'qualification': qual,
            'training_hours': np.clip(rng.normal(30, 12, n), 0, 80), 'satisfaction': rng.integers(1, 5, n).astype(float),
            'perf_score': np.clip(0.45 + 0.012 * exp_ + 0.04 * (qual - 3) + 0.1 * rng.standard_normal(n), 0.05, 1.0)})
    prof = make_role_profiles(emp['role'], K, rng)
    emp['experience'] = emp['experience'].clip(0.3, 40)
    skills = np.stack([prof[r] for r in emp['role']]) * 0.75 + 0.15 * np.minimum(emp['experience'].values[:, None] / 15, 1) \
             + 0.10 * rng.standard_normal((n, K))
    for k in range(K): emp[f'skill_{k}'] = np.clip(skills[:, k], 0, 1)
    emp['capacity_h'] = np.clip(rng.normal(6.0, 0.6, n), 4.5, 7.5)       # productive hours/day
    emp['attendance'] = np.clip(rng.beta(20, 2, n), 0.7, 1.0)
    emp['leaves'] = rng.poisson(8, n).astype(float)
    if 'turnover' not in emp:
        emp['turnover'] = (rng.random(n) < sigmoid(-2.0 + 1.5 * (1 - emp['satisfaction'] / 4) + 2 * (1 - emp['attendance']))).astype(int)
    emp.insert(0, 'emp_id', np.arange(n))
    return emp, prof

def build_tasks(cfg, emp, prof, rng):
    T, K = cfg['N_TASKS'], cfg['N_SKILLS']
    depts = sorted(emp['dept'].unique()); dp = emp['dept'].value_counts(normalize=True)[depts].values
    dept = rng.choice(depts, T, p=dp); ttype = rng.choice(TASK_TYPES, T)
    arrival = np.sort(rng.uniform(0, cfg['HORIZON'], T))
    dprof = {d: np.mean([prof[r] for r in emp.loc[emp.dept == d, 'role'].unique()], 0) for d in depts}
    req = np.zeros((T, K))
    for i in range(T):
        p = dprof[dept[i]] + 0.05; p = p / p.sum()
        k = rng.choice(K, rng.integers(2, 4), replace=False, p=p); req[i, k] = rng.uniform(0.4, 1.0, len(k))
    est = np.clip(rng.lognormal(math.log(22), 0.55, T), 4, 120)
    prio = rng.integers(1, 4, T).astype(float); cx = rng.integers(1, 6, T).astype(float)
    slack = rng.uniform(1.3, 3.2, T) - 0.15 * (prio - 1)
    week = (arrival // 7).astype(int)
    project = pd.factorize(pd.Series(dept) + '|' + pd.Series(ttype) + '|' + pd.Series(week.astype(str)))[0]
    tasks = pd.DataFrame({'task_id': np.arange(T), 'arrival': arrival, 'task_type': ttype, 'dept': dept, 'priority': prio,
                          'est_duration': est, 'deadline_h': est * slack, 'complexity': cx, 'project': project})
    for k in range(K): tasks[f'req_{k}'] = req[:, k]
    return tasks

In [ ]:
class World:
    """Latent organisational ground truth (used only by the simulator / allocation environment)."""
    def __init__(s, emp, tasks, K, rng):
        s.E, s.T, s.K = len(emp), len(tasks), K
        dnames = sorted(emp['dept'].unique()); s.dept_names = dnames; dmap = {d: i for i, d in enumerate(dnames)}
        s.D = len(dnames)
        s.dept_id = emp['dept'].map(dmap).values; s.tdept = tasks['dept'].map(dmap).values
        s.skill_true = emp[[f'skill_{k}' for k in range(K)]].values.copy()
        s.perf_true = emp['perf_score'].values.copy(); s.cap = emp['capacity_h'].values.copy()
        s.req = tasks[[f'req_{k}' for k in range(K)]].values.copy()
        s.est = tasks['est_duration'].values.copy(); s.deadline = tasks['deadline_h'].values.copy()
        s.cx_true = tasks['complexity'].values.copy(); s.prio = tasks['priority'].values.copy()
        s.arrival = tasks['arrival'].values.copy()
        s.dept_members = [np.where(s.dept_id == d)[0] for d in range(s.D)]
        s.noise = rng.standard_normal((s.T, s.E, 3)).astype(np.float32)     # common random numbers
        s.obs = None                                                         # observed (imputed) arrays, set in Stage 2
    def skill_match(s, e, t, skill=None):
        sk = s.skill_true if skill is None else skill
        r = s.req[t]; return (r * sk[e]).sum(-1) / r.sum(-1)
    def outcome(s, e, t, load_h, drift=0.0, noise=None):
        nz = s.noise[t, e] if noise is None else noise
        sm = s.skill_match(e, t); perf = s.perf_true[e]; cx = (s.cx_true[t] - 1) / 4
        lr = load_h / (s.cap[e] * 5); over = np.clip(lr - 0.7, 0, None)
        ability = 0.55 * sm + 0.45 * perf
        speed = np.clip(0.45 + 1.0 * ability - 0.45 * over - 0.30 * cx - drift, 0.2, 2.0)
        turn = (0.5 * load_h + s.est[t] / speed) * np.exp(0.18 * nz[..., 0])
        quality = np.clip(0.35 + 0.7 * ability - 0.25 * cx - 0.3 * over - 0.5 * drift + 0.08 * nz[..., 1], 0, 1)
        dl = s.deadline[t]
        suit_lat = sigmoid(6 * (ability - 0.5) - 2.5 * over - 1.5 * cx + 0.5)
        return dict(turn=turn, quality=quality, delay=(turn > dl).astype(float),
                    complete=((turn <= 1.5 * dl) & (quality >= 0.45)).astype(float),
                    suit=(suit_lat + 0.1 * nz[..., 1] > 0.5).astype(float),
                    prod=np.clip(s.est[t] / turn, 0, 1.5) * quality, sm=sm)
    def attach_observed(s, emp_imp, task_imp, K):
        s.obs = dict(skill=emp_imp[[f'skill_{k}' for k in range(K)]].values, perf=emp_imp['perf_score'].values,
                     exp=emp_imp['experience'].values, cx=task_imp['complexity'].values)
    def scal(s, e, t, lr_before, hd):
        """Interpretable pair-level inputs (observed/imputed values only) -> (n, 9)."""
        o = s.obs; e = np.asarray(e); n = len(e); t = np.broadcast_to(np.asarray(t), (n,))
        sm = (s.req[t] * o['skill'][e]).sum(-1) / s.req[t].sum(-1)
        return np.stack([sm, o['perf'][e], o['exp'][e], lr_before, o['cx'][t], s.deadline[t] / s.est[t],
                         s.prio[t], s.est[t], hd], -1).astype(np.float32)

class EmpState:
    """Dynamic workforce state: backlog (hours), recent outcome history (only finished tasks are visible)."""
    def __init__(s, world, L, init_load=None):
        s.w, s.L, s.t = world, L, 0.0
        s.load = np.zeros(world.E) if init_load is None else init_load.copy(); s.hist = [[] for _ in range(world.E)]
    def advance(s, now):
        dt = max(now - s.t, 0.0); s.load = np.maximum(s.load - s.w.cap * dt, 0.0); s.t = now
    def load_ratio(s, idx=None):
        lr = s.load / (s.w.cap * 5); return lr if idx is None else lr[idx]
    def seq(s, e_idx, now):
        L = s.L; S = np.zeros((len(e_idx), L, 6), np.float32); hd = np.full(len(e_idx), 0.3, np.float32)
        for i, e in enumerate(e_idx):
            vis = [h for h in s.hist[e] if h[0] <= now]
            if not vis: continue
            vis = sorted(vis, key=lambda h: h[0])[-L:]
            for j, h in enumerate(vis):
                S[i, L - len(vis) + j, :5] = h[1:6]; S[i, L - len(vis) + j, 5] = 1
            hd[i] = np.mean([h[1] for h in vis])
        return S, hd
    def assign(s, e, t, out, now):
        w = s.w; lr_b = s.load[e] / (w.cap[e] * 5); s.load[e] += w.est[t]
        lr = s.load / (w.cap * 5); la = lr[e]; dm = lr[w.dept_members[w.dept_id[e]]].mean(); nz = w.noise[t, e]
        info = dict(lr_before=lr_b, lr_after=la, risk=float(la + 0.12 * nz[2] > 1.5), imbalance=float(la - dm > 0.8),
                    delay=float(out['delay']), complete=float(out['complete']), suit=float(out['suit']),
                    prod=float(out['prod']), quality=float(out['quality']), turn=float(out['turn']))
        s.hist[e].append((now + out['turn'] / w.cap[e], info['delay'], info['prod'], info['quality'], lr_b, math.log(w.est[t]) / 5))
        return info

def simulate_history(world, cfg, rng):
    """Historical (partly random, load/skill-aware) assignment policy -> assignment log + temporal histories."""
    st = EmpState(world, cfg['SEQ_LEN']); rows, seqs = [], []
    for t in range(world.T):                                   # tasks are sorted by arrival
        now = world.arrival[t]; st.advance(now)
        cands = np.arange(world.E) if rng.random() < 0.12 else world.dept_members[world.tdept[t]]
        util = -1.2 * st.load_ratio(cands) + 1.0 * world.skill_match(cands, t) + 0.8 * rng.standard_normal(len(cands))
        p = np.exp((util - util.max()) / 0.5); p /= p.sum(); e = int(rng.choice(cands, p=p))
        sq, hd = st.seq([e], now); lr_b = st.load_ratio([e])[0]
        out = world.outcome(e, t, st.load[e]); info = st.assign(e, t, out, now)
        rows.append(dict(task_id=t, emp_id=e, arrival=now, lr_before=lr_b, hd=hd[0], **{k: info[k] for k in
                    ['delay', 'complete', 'suit', 'risk', 'imbalance', 'prod', 'quality', 'turn', 'lr_after']}))
        seqs.append(sq[0])
    return pd.DataFrame(rows), np.stack(seqs)

In [ ]:
def inject_missing(df, cols, frac, rng):
    df = df.copy()
    for c in cols: df.loc[rng.random(len(df)) < frac, c] = np.nan
    return df

def run_stage1(cfg, seed):
    rng = np.random.default_rng(seed)
    emp, prof = build_employees(cfg, rng); tasks = build_tasks(cfg, emp, prof, rng)
    world = World(emp, tasks, cfg['N_SKILLS'], rng)
    log, seqs = simulate_history(world, cfg, rng)
    # raw logs are noisy: inject missing values + duplicated log lines, then clean (Stage 1 processing)
    emp_obs = inject_missing(emp, ['experience', 'training_hours', 'perf_score', 'satisfaction', 'attendance'], 0.04, rng)
    task_obs = inject_missing(tasks, ['complexity'], 0.04, rng)
    dup = log.sample(frac=0.01, random_state=seed); log_raw = pd.concat([log, dup]).sort_values('task_id', kind='stable')
    report = dict(emp_missing=emp_obs.isna().sum()[lambda s: s > 0].to_dict(), task_missing=task_obs.isna().sum()[lambda s: s > 0].to_dict(),
                  duplicate_rows=int(log_raw.duplicated(subset=['task_id', 'emp_id']).sum()))
    keep = ~log_raw.duplicated(subset=['task_id', 'emp_id'], keep='first').values
    log_clean = log_raw[keep].reset_index(drop=True); assert len(log_clean) == len(log)
    return dict(emp=emp, tasks=tasks, emp_obs=emp_obs, task_obs=task_obs, world=world, log=log_clean, seqs=seqs, report=report)

In [ ]:
S = run_stage1(CFG, CFG['SEED'])
print('Data-quality report (before cleaning):', S['report'])
print('employees', S['emp'].shape, '| tasks', S['tasks'].shape, '| assignment log', S['log'].shape, '| temporal histories', S['seqs'].shape)
display(S['emp_obs'].head(3)); display(S['task_obs'].head(3)); display(S['log'].head(3))

## Stage 2 — HR Data Preprocessing
**KNN imputation → Robust scaling → Target encoding → Quantile transformation → Temporal feature construction.**  
Numeric variables are imputed/normalised, categorical variables (role, department, task type) are target-encoded using *training rows only*, and past task outcomes are turned into workload/productivity trend features (only tasks already finished at assignment time are visible – no leakage).

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler, QuantileTransformer

K = CFG['N_SKILLS']; SK = [f'skill_{k}' for k in range(K)]; RQ = [f'req_{k}' for k in range(K)]


class SmoothTargetEncoder:
    """Smoothed mean target encoding, fitted on training rows only."""
    def __init__(self, m=20): self.m = m
    def fit(self, cats, y):
        self.g = y.mean(); df = pd.DataFrame({'c': cats, 'y': y}).groupby('c')['y'].agg(['sum', 'count'])
        self.map = ((df['sum'] + self.m * self.g) / (df['count'] + self.m)).to_dict(); return self
    def transform(self, cats): return pd.Series(cats).map(self.map).fillna(self.g).values


def run_stage2(S, cfg):
    emp_obs, task_obs, world, log, seqs = S['emp_obs'], S['task_obs'], S['world'], S['log'], S['seqs']
    L = cfg['SEQ_LEN']; N = len(log)
    # temporal split by arrival time (no shuffling)
    ntr, nva = int(0.65 * N), int(0.15 * N)
    split = np.array(['train'] * ntr + ['val'] * nva + ['test'] * (N - ntr - nva))
    # 1) KNN imputation of numeric columns
    e_num = ['experience', 'qualification', 'training_hours', 'capacity_h', 'attendance', 'leaves', 'perf_score', 'satisfaction'] + SK
    t_num = ['priority', 'est_duration', 'deadline_h', 'complexity'] + RQ
    emp_imp = emp_obs.copy(); emp_imp[e_num] = KNNImputer(n_neighbors=5).fit_transform(emp_obs[e_num])
    task_imp = task_obs.copy(); task_imp[t_num] = KNNImputer(n_neighbors=5).fit_transform(task_obs[t_num])
    world.attach_observed(emp_imp, task_imp, cfg['N_SKILLS'])
    # 2) target encoding of categoricals (target = delay, train rows only)
    tr = log[split == 'train']
    te = {}
    for name, cats in [('role', emp_imp['role'].values[tr.emp_id]), ('edept', emp_imp['dept'].values[tr.emp_id]),
                       ('ttype', task_imp['task_type'].values[tr.task_id]), ('tdept', task_imp['dept'].values[tr.task_id])]:
        te[name] = SmoothTargetEncoder().fit(cats, tr['delay'].values)
    # 3) robust scaling -> quantile transformation (numeric node features)
    def scale_block(df, cols):
        x = RobustScaler().fit_transform(df[cols].values)
        return QuantileTransformer(n_quantiles=min(100, len(df)), output_distribution='normal', random_state=0).fit_transform(x)
    emp_X = np.hstack([scale_block(emp_imp, e_num), te['role'].transform(emp_imp['role'])[:, None], te['edept'].transform(emp_imp['dept'])[:, None]])
    t_num2 = ['priority', 'est_duration', 'deadline_h', 'complexity']
    tdf = task_imp.copy(); tdf['est_duration'] = np.log1p(tdf['est_duration']); tdf['deadline_h'] = np.log1p(tdf['deadline_h'])
    task_X = np.hstack([scale_block(tdf, t_num2), task_imp[RQ].values, te['ttype'].transform(task_imp['task_type'])[:, None],
                        te['tdept'].transform(task_imp['dept'])[:, None]])
    # 4) temporal feature construction (from visible past outcomes only)
    vis = seqs[:, :, 5:6]; cnt = vis.sum((1, 2)).clip(min=1)
    temporal = pd.DataFrame({'roll_delay': seqs[:, :, 0].sum(1) / cnt, 'roll_prod': seqs[:, :, 1].sum(1) / cnt,
                             'roll_quality': seqs[:, :, 2].sum(1) / cnt, 'n_recent': vis.sum((1, 2)),
                             'workload_trend': seqs[:, -1, 3] - seqs[:, 0, 3]})
    e_idx, t_idx = log['emp_id'].values, log['task_id'].values
    scal = world.scal(e_idx, t_idx, log['lr_before'].values, log['hd'].values)
    Y = np.stack([log['complete'], log['risk'], log['delay'], log['suit'], log['imbalance'], log['prod']], 1).astype(np.float32)
    flat = np.hstack([emp_X[e_idx], task_X[t_idx], scal, temporal.values]).astype(np.float32)   # tabular matrix for baselines
    return dict(emp_X=emp_X.astype(np.float32), task_X=task_X.astype(np.float32), e_idx=e_idx, t_idx=t_idx, scal=scal, seqs=seqs,
                Y=Y, split=split, flat=flat, temporal=temporal, emp_imp=emp_imp, task_imp=task_imp, log=log, world=world)

In [ ]:
D = run_stage2(S, CFG)
print('node features  employees', D['emp_X'].shape, ' tasks', D['task_X'].shape, '| pair table', D['flat'].shape)
print('temporal split:', pd.Series(D['split']).value_counts().to_dict())
print('positive rate per target:', dict(zip(TARGETS[:5], D['Y'][:, :5].mean(0).round(3))))
print('missing values left in model inputs:', int(np.isnan(D['flat']).sum()))
display(pd.DataFrame(D['scal'], columns=PAIR_FEATS).describe().T.round(3)); display(D['temporal'].describe().T.round(3))

## Stage 3 — Employee–Task Relational Representation (Contribution 1: **TERRN**)
**Heterogeneous graph** (employee, task, skill, department nodes; employee–task, employee–skill, task–skill, employee–department, employee–employee collaboration edges) → **relation-aware attention** → **hypergraph message passing** (project teams as hyperedges) → **temporal message passing** (GRU over each employee's recent outcomes).  
Output: unified employee–task relational embeddings `z = [capability ; demand]`.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(CFG['SEED']); torch.set_num_threads(max(1, os.cpu_count() or 1))
tt = lambda a, dt=torch.float32: torch.as_tensor(np.asarray(a), dtype=dt)
SQ2 = math.sqrt(2.0)

# ----------------------------------------------------------------------------------------------------------
# Tensors for pair-level learning (temporal split already encoded in D['split'])
# ----------------------------------------------------------------------------------------------------------
def make_tensors(D):
    tr = D['split'] == 'train'
    mu = D['scal'][tr].mean(0); sd = D['scal'][tr].std(0) + 1e-6
    y = D['Y'].copy(); pm, ps = float(y[tr, 5].mean()), float(y[tr, 5].std()); y[:, 5] = (y[:, 5] - pm) / ps
    return dict(e=torch.as_tensor(D['e_idx']).long(), t=torch.as_tensor(D['t_idx']).long(), scal_raw=tt(D['scal']),
                scal=tt((D['scal'] - mu) / sd).clamp(-5, 5), seq=tt(D['seqs']), y=tt(y), mu=tt(mu), sd=tt(sd), pm=pm, ps=ps,
                idx={k: np.where(D['split'] == k)[0] for k in ['train', 'val', 'test']})

def mt_loss(logits, y):
    """5 binary targets (BCE) + productivity regression (MSE, standardised)."""
    return F.binary_cross_entropy_with_logits(logits[:, :5], y[:, :5]) + 0.5 * F.mse_loss(logits[:, 5], y[:, 5])

In [ ]:
# ----------------------------------------------------------------------------------------------------------
# Stage 3 - TERRN: heterogeneous graph + relation-aware attention + hypergraph + temporal message passing
# ----------------------------------------------------------------------------------------------------------
def build_graph(D):
    """Dense typed adjacency matrices. Assignment edges / hyperedges use TRAIN-period assignments only."""
    w, log, tr = D['world'], D['log'], D['split'] == 'train'; o = w.obs
    E, T, Dn = w.E, w.T, w.D
    A_et = np.zeros((E, T), np.float32); A_et[log.emp_id.values[tr], log.task_id.values[tr]] = 1
    proj = D['task_imp']['project'].values; team = {}
    for e, t in zip(log.emp_id.values[tr], log.task_id.values[tr]): team.setdefault(proj[t], set()).add(e)
    teams = [sorted(v) for v in team.values() if len(v) >= 2]
    H = np.zeros((E, len(teams)), np.float32)
    for j, m in enumerate(teams): H[m, j] = 1
    dv, de = H.sum(1), H.sum(0)
    P = (H / np.maximum(dv, 1)[:, None]) @ (H / np.maximum(de, 1)[None, :]).T          # Dv^-1 H De^-1 H^T
    C = H @ H.T; A_ee = ((C >= 2) & ~np.eye(E, dtype=bool)).astype(np.float32)          # collaboration edges
    g = dict(xe=D['emp_X'], xt=D['task_X'], es=np.where(o['skill'] > 0.2, o['skill'], 0.0), ts=w.req, et=A_et, ee=A_ee,
             ed=np.eye(Dn)[w.dept_id], td=np.eye(Dn)[w.tdept], P=P)
    G = {k: tt(v) for k, v in g.items()}; G['n_hyperedges'] = len(teams); return G

class RelAttn(nn.Module):
    """Relation-specific multi-head attention over a dense (dst x src) weighted adjacency."""
    def __init__(s, d, heads=2):
        super().__init__(); s.h, s.dh = heads, d // heads
        s.q, s.k, s.v, s.o = nn.Linear(d, d), nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False), nn.Linear(d, d)
        s.rel_bias = nn.Parameter(torch.zeros(heads))
    def forward(s, xd, xs, A):
        Nd, Ns = A.shape
        q = s.q(xd).view(Nd, s.h, s.dh); k = s.k(xs).view(Ns, s.h, s.dh); v = s.v(xs).view(Ns, s.h, s.dh)
        sc = torch.einsum('dhc,shc->hds', q, k) / math.sqrt(s.dh) + s.rel_bias.view(-1, 1, 1)
        mask = (A > 0); sc = sc + torch.log(A.clamp_min(1e-6)).unsqueeze(0)
        att = torch.softmax(sc.masked_fill(~mask.unsqueeze(0), -1e9), -1) * mask.unsqueeze(0)
        return s.o(torch.einsum('hds,shc->dhc', att, v).reshape(Nd, -1))

class HeteroLayer(nn.Module):
    REL = ['e_s', 'e_t', 'e_e', 'e_d', 't_s', 't_d', 's_e', 's_t', 'd_e']
    def __init__(s, d):
        super().__init__()
        s.r = nn.ModuleDict({k: RelAttn(d) for k in s.REL}); s.sl = nn.ModuleDict({k: nn.Linear(d, d) for k in 'etsd'})
        s.nm = nn.ModuleDict({k: nn.LayerNorm(d) for k in 'etsd'})
    def forward(s, X, G):
        e = s.sl['e'](X['e']) + s.r['e_s'](X['e'], X['s'], G['es']) + s.r['e_t'](X['e'], X['t'], G['et']) \
            + s.r['e_e'](X['e'], X['e'], G['ee']) + s.r['e_d'](X['e'], X['d'], G['ed'])
        t = s.sl['t'](X['t']) + s.r['t_s'](X['t'], X['s'], G['ts']) + s.r['t_d'](X['t'], X['d'], G['td'])
        sk = s.sl['s'](X['s']) + s.r['s_e'](X['s'], X['e'], G['es'].T) + s.r['s_t'](X['s'], X['t'], G['ts'].T)
        d = s.sl['d'](X['d']) + s.r['d_e'](X['d'], X['e'], G['ed'].T)
        new = {'e': e, 't': t, 's': sk, 'd': d}
        return {k: s.nm[k](X[k] + F.gelu(v)) for k, v in new.items()}

class HyperConv(nn.Module):
    """Hypergraph convolution: X' = sigma(Dv^-1 H De^-1 H^T X Theta) - team/project-level higher-order message passing."""
    def __init__(s, d): super().__init__(); s.theta = nn.Linear(d, d)
    def forward(s, x, P): return F.gelu(s.theta(P @ x))

class TERRN(nn.Module):
    def __init__(s, d_e, d_t, n_skill, n_dept, d=32, layers=2, seq_f=6, scal_f=9, use_hyper=True, use_temporal=True):
        super().__init__(); s.d, s.use_hyper, s.use_temporal = d, use_hyper, use_temporal
        s.pe, s.pt = nn.Linear(d_e, d), nn.Linear(d_t, d)
        s.skill_emb, s.dept_emb = nn.Embedding(n_skill, d), nn.Embedding(n_dept, d)
        s.layers = nn.ModuleList([HeteroLayer(d) for _ in range(layers)]); s.hyp = nn.ModuleList([HyperConv(d) for _ in range(layers)])
        s.gru = nn.GRU(seq_f, d, batch_first=True)                       # temporal message passing over past task outcomes
        s.fa = nn.Sequential(nn.Linear(3 * d + scal_f, d), nn.GELU(), nn.Linear(d, d // 2))     # capability half  a
        s.fb = nn.Sequential(nn.Linear(2 * d + scal_f, d), nn.GELU(), nn.Linear(d, d // 2))     # demand half      b
    def node_embed(s, G):
        X = {'e': s.pe(G['xe']), 't': s.pt(G['xt']), 's': s.skill_emb.weight, 'd': s.dept_emb.weight}
        for hl, hc in zip(s.layers, s.hyp):
            X = hl(X, G)
            if s.use_hyper: X['e'] = X['e'] + hc(X['e'], G['P'])
        return X['e'], X['t']
    def pair_embed(s, Ee, Te, e_idx, t_idx, scal, seq):
        e, t = Ee[e_idx], Te[t_idx]
        h = s.gru(seq)[1][-1] if s.use_temporal else torch.zeros_like(e)
        a = s.fa(torch.cat([e, h, e * t, scal], -1)); b = s.fb(torch.cat([t, (e - t).abs(), scal], -1))
        return torch.cat([a, b], -1)                                      # z = [capability ; demand]

In [ ]:
TD = make_tensors(D); G = build_graph(D)
print('graph: employees', G['xe'].shape[0], '| tasks', G['xt'].shape[0], '| train assignment edges', int(G['et'].sum()),
      '| collaboration edges', int(G['ee'].sum()), '| hyperedges (project teams)', G['n_hyperedges'])

## Stage 4 — Reflection-Equivariant HR Feature Learning (Contribution 2: **RE-WFE**)
**Reflection-equivariant layers → group-equivariant representation learning → symmetry-preserving feature transformation → (equivariant) layer normalisation.**  
TERRN and RE-WFE are trained jointly with a classical multi-task head (this also gives the *classical TERRN+RE-WFE* reference); the resulting embeddings feed the quantum stages.

In [ ]:
# ----------------------------------------------------------------------------------------------------------
# Stage 4 - RE-WFE: Z2 reflection-equivariant encoder.  Reflection R swaps the capability and demand halves of z.
#   even channel  x+ = (a+b)/sqrt2  (symmetric alignment),   odd channel x- = (a-b)/sqrt2  (capability-demand gap, flips sign)
#   R acts as (x+, x-) -> (x+, -x-).  Block-diagonal linear maps + invariant gates + equivariant LayerNorm keep this exact.
# ----------------------------------------------------------------------------------------------------------
class EquivLayer(nn.Module):
    def __init__(s, d2):
        super().__init__(); s.Wp = nn.Linear(d2, d2); s.Wm = nn.Linear(d2, d2, bias=False); s.gate = nn.Linear(2 * d2, d2)
    def forward(s, xp, xm):
        g = torch.sigmoid(s.gate(torch.cat([xp, xm.abs()], -1)))          # invariant gate (uses |x-|)
        return F.gelu(s.Wp(xp)), s.Wm(xm) * g

class EquivLayerNorm(nn.Module):
    def __init__(s, d2): super().__init__(); s.gp, s.gm, s.eps = nn.Parameter(torch.ones(d2)), nn.Parameter(torch.ones(d2)), 1e-5
    def forward(s, xp, xm):
        xp = xp - xp.mean(-1, keepdim=True)                                # centring only on the even part
        sc = torch.sqrt((xp.pow(2).mean(-1, keepdim=True) + xm.pow(2).mean(-1, keepdim=True)) / 2 + s.eps)
        return s.gp * xp / sc, s.gm * xm / sc

class REWFE(nn.Module):
    def __init__(s, d=32, layers=2, equivariant=True):
        super().__init__(); s.d, s.equivariant = d, equivariant; d2 = d // 2
        if equivariant:
            s.L = nn.ModuleList([EquivLayer(d2) for _ in range(layers)]); s.N = nn.ModuleList([EquivLayerNorm(d2) for _ in range(layers)])
        else:   # ablation: ordinary MLP encoder of the same width
            s.mlp = nn.ModuleList([nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.LayerNorm(d)) for _ in range(layers)])
    def forward(s, z):
        if not s.equivariant:
            for m in s.mlp: z = z + m(z)
            return z
        a, b = z.chunk(2, -1); xp, xm = (a + b) / SQ2, (a - b) / SQ2
        for L, N in zip(s.L, s.N):
            hp, hm = L(xp, xm); xp, xm = N(xp + hp, xm + hm)
        return torch.cat([xp, xm], -1)

@torch.no_grad()
def check_rewfe_equivariance(rewfe, d=32):
    z = torch.randn(64, d); R_in = lambda z: torch.cat([z[:, d // 2:], z[:, :d // 2]], -1)
    R_out = lambda o: torch.cat([o[:, :d // 2], -o[:, d // 2:]], -1)
    rewfe.eval(); return (rewfe(R_in(z)) - R_out(rewfe(z))).abs().max().item()

class Stage34(nn.Module):
    def __init__(s, d_e, d_t, n_skill, n_dept, d=32, use_hyper=True, use_temporal=True, rewfe_equiv=True):
        super().__init__(); s.terrn = TERRN(d_e, d_t, n_skill, n_dept, d, use_hyper=use_hyper, use_temporal=use_temporal)
        s.rewfe = REWFE(d, equivariant=rewfe_equiv)
        s.head = nn.Sequential(nn.Linear(d, 64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64, 6))   # classical head (pre-training / baseline)
    def forward(s, G, e, t, scal, seq, cache=None):
        Ee, Te = s.terrn.node_embed(G) if cache is None else cache
        zr = s.rewfe(s.terrn.pair_embed(Ee, Te, e, t, scal, seq)); return s.head(zr), zr

def train_stage34(model, G, TD, epochs, lr=2e-3, bs=512, verbose=True):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    tri, vai = TD['idx']['train'], torch.as_tensor(TD['idx']['val']); best, best_state, hist = 1e9, None, []
    for ep in range(epochs):
        model.train(); perm = np.random.permutation(tri); tl = []
        for i in range(0, len(perm), bs):
            b = torch.as_tensor(perm[i:i + bs])
            logits, _ = model(G, TD['e'][b], TD['t'][b], TD['scal'][b], TD['seq'][b])
            loss = mt_loss(logits, TD['y'][b]); opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); tl.append(loss.item())
        model.eval()
        with torch.no_grad():
            lv, _ = model(G, TD['e'][vai], TD['t'][vai], TD['scal'][vai], TD['seq'][vai]); vl = mt_loss(lv, TD['y'][vai]).item()
        hist.append((float(np.mean(tl)), vl))
        if vl < best: best, best_state = vl, copy.deepcopy(model.state_dict())
        if verbose and (ep % 5 == 0 or ep == epochs - 1): print(f'  [TERRN+RE-WFE] ep {ep:3d} train {np.mean(tl):.4f} val {vl:.4f}')
    model.load_state_dict(best_state); model.eval(); return hist

@torch.no_grad()
def extract_embeddings(model, G, TD):
    model.eval(); cache = model.terrn.node_embed(G)
    _, Z = model(G, TD['e'], TD['t'], TD['scal'], TD['seq'], cache=cache); return Z, cache

In [ ]:
print('RE-WFE reflection-equivariance error (untrained):', check_rewfe_equivariance(REWFE(CFG['D'])))
S34_CACHE = {}
torch.manual_seed(0); np.random.seed(0)
s34 = Stage34(D['emp_X'].shape[1], D['task_X'].shape[1], D['world'].K, D['world'].D, CFG['D'])
h34 = train_stage34(s34, G, TD, EP['graph'])
Z, cache = extract_embeddings(s34, G, TD)
S34_CACHE[(True, True, True, EP['graph'], 0)] = (s34, h34, Z, cache)
print('RE-WFE reflection-equivariance error (trained):', check_rewfe_equivariance(s34.rewfe))
print('embedding matrix:', tuple(Z.shape))

## Stage 5 — Quantum Task-State Encoding
**PCA-based quantum feature compression → angle encoding → amplitude encoding → parameterised circuits.**  
The 32-d reflection-equivariant embedding is compressed to `n_qubits` principal components (angle encoding, scaled to [0, π]) or to a normalised 2ⁿ-amplitude vector (amplitude encoding, used in the ablation).

In [ ]:
from sklearn.decomposition import PCA

# ----------------------------------------------------------------------------------------------------------
# Stage 5 - Quantum task-state encoding (PCA compression -> angle / amplitude encoding)
# ----------------------------------------------------------------------------------------------------------
class QPre:
    def __init__(s, n_qubits, mode='angle'): s.n, s.mode = n_qubits, mode
    def fit(s, Z):
        k = s.n if s.mode == 'angle' else min(2 ** s.n, Z.shape[1])
        s.pca = PCA(n_components=k, random_state=0).fit(Z); u = s.pca.transform(Z)
        s.mean, s.comp = tt(s.pca.mean_), tt(s.pca.components_)
        s.lo, s.hi = tt(np.percentile(u, 1, 0)), tt(np.percentile(u, 99, 0)); s.explained = float(s.pca.explained_variance_ratio_.sum()); return s
    def __call__(s, Z):
        u = (Z - s.mean) @ s.comp.T
        if s.mode == 'angle': return ((u - s.lo) / (s.hi - s.lo + 1e-8)).clamp(0, 1) * math.pi        # angle encoding in [0, pi]
        u = F.pad(u, (0, 2 ** s.n - u.shape[1])); return u / (u.norm(dim=-1, keepdim=True) + 1e-8)   # amplitude encoding

In [ ]:
qpre = QPre(CFG['N_QUBITS'], CFG['ENCODING']).fit(Z[TD['idx']['train']].numpy())
Xq = qpre(Z)
print(f"PCA -> {CFG['N_QUBITS']} qubits, explained variance {qpre.explained:.3f}; encoded range [{Xq.min():.2f}, {Xq.max():.2f}]")
plt.figure(figsize=(4, 2.5)); plt.bar(range(CFG['N_QUBITS']), qpre.pca.explained_variance_ratio_); plt.xlabel('qubit / PC'); plt.ylabel('explained var.'); plt.tight_layout(); plt.show()

## Stage 6 — REQNN-Based HR Prediction (Contribution 3: **REQ-WN**)
**Reflection-equivariant quantum neural network → variational quantum layers → data re-uploading → parameterised entanglement → measurement.**  
One circuit serves all six outputs (task-completion probability, employee workload risk, task-delay probability, employee–task suitability, workload imbalance, productivity level) through measured ⟨Zᵢ⟩ and ⟨ZᵢZⱼ⟩ expectation values and a linear readout. Evaluation helpers and the experiment runner (also used for the ablation) are defined first.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score,
                             matthews_corrcoef, mean_absolute_error, r2_score)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

# Prediction metrics (thresholds are chosen on the validation block, metrics reported on the temporal test block)
# ----------------------------------------------------------------------------------------------------------
def eval_multitask(prob, prod_pred, Y, idx):
    """prob: (N,5) probabilities for all rows, prod_pred: (N,) productivity in raw units."""
    rows = []
    for j, name in enumerate(TARGETS[:5]):
        yv, pv = Y[idx['val'], j], prob[idx['val'], j]; ts = np.linspace(0.05, 0.95, 91)
        th = ts[int(np.argmax([f1_score(yv, pv >= t, zero_division=0) for t in ts]))]
        yt, pt = Y[idx['test'], j], prob[idx['test'], j]; yh = pt >= th
        ok = len(np.unique(yt)) > 1
        rows.append(dict(target=name, Accuracy=accuracy_score(yt, yh), Precision=precision_score(yt, yh, zero_division=0),
                         Recall=recall_score(yt, yh, zero_division=0), F1=f1_score(yt, yh, zero_division=0),
                         ROC_AUC=roc_auc_score(yt, pt) if ok else np.nan, PR_AUC=average_precision_score(yt, pt) if ok else np.nan,
                         MCC=matthews_corrcoef(yt, yh)))
    df = pd.DataFrame(rows).set_index('target')
    reg = dict(MAE=mean_absolute_error(Y[idx['test'], 5], prod_pred[idx['test']]), R2=r2_score(Y[idx['test'], 5], prod_pred[idx['test']]))
    return df, reg

def summarize(df, reg): return {**df.mean().round(4).to_dict(), 'prod_MAE': round(reg['MAE'], 4), 'prod_R2': round(reg['R2'], 4)}

def baseline_predictions(name, X, Y, idx):
    tr = idx['train']; P = np.zeros((len(X), 5)); t0 = time.time()
    mk = {'LogReg': lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=0.5)),
          'RandForest': lambda: RandomForestClassifier(200, min_samples_leaf=3, n_jobs=-1, random_state=0),
          'HistGB': lambda: HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, random_state=0),
          'MLP': lambda: make_pipeline(StandardScaler(), MLPClassifier((64, 32), max_iter=300, early_stopping=True, random_state=0))}[name]
    for j in range(5): P[:, j] = mk().fit(X[tr], Y[tr, j]).predict_proba(X)[:, 1]
    reg = HistGradientBoostingRegressor(max_iter=150, random_state=0).fit(X[tr], Y[tr, 5]); pr = reg.predict(X)
    return P, pr, time.time() - t0

In [ ]:
# ----------------------------------------------------------------------------------------------------------
# Stage 6 - REQ-WN: exact state-vector simulation in PyTorch (differentiable, no external quantum library)
#   Qubit 0 is the most significant bit.  Reflection = mirror of the qubit register (q <-> n-1-q).
#   Per layer:  data (re-)uploading RY(w*x_q)  ->  trainable Rot = RZ RY RZ  ->  parameterised ZZ-entangler ring.
#   Parameters are tied across mirror pairs, ring-ZZ parameters are tied across mirror pairs of edges, ZZ gates commute,
#   hence the circuit is EXACTLY reflection-equivariant:  <Z_i>(reverse(x)) = <Z_{n-1-i}>(x).
# ----------------------------------------------------------------------------------------------------------
def _mat(a, b, c, d): return torch.stack([torch.stack([a, b], -1), torch.stack([c, d], -1)], -2)
def RY(t):
    c, s_ = torch.cos(t / 2), torch.sin(t / 2); z = torch.zeros_like(c); C = torch.complex
    return _mat(C(c, z), C(-s_, z), C(s_, z), C(c, z))
def RZ(t):
    c, s_ = torch.cos(t / 2), torch.sin(t / 2); z = torch.zeros_like(c); C = torch.complex
    return _mat(C(c, -s_), C(z, z), C(z, z), C(c, s_))

def apply_1q(state, U, q, n):
    B = state.shape[0]; st = state.reshape(B, 2 ** q, 2, 2 ** (n - q - 1))
    if U.dim() == 2: U = U.unsqueeze(0).expand(B, -1, -1)
    return torch.einsum('bij,bajc->baic', U, st).reshape(B, -1)

class REQWN(nn.Module):
    def __init__(s, n_qubits=6, layers=3, n_out=6, encoding='angle', reupload=True, equivariant=True):
        super().__init__(); n = n_qubits; s.n, s.L, s.enc, s.reup, s.eq = n, layers, encoding, reupload, equivariant
        idx = torch.arange(2 ** n); bits = torch.stack([(idx >> (n - 1 - q)) & 1 for q in range(n)], 1).float()
        z = 1 - 2 * bits; pairs = [(q, (q + 1) % n) for q in range(n)] if n > 2 else [(0, 1)]
        s.register_buffer('zs', z); s.register_buffer('zz', torch.stack([z[:, a] * z[:, b] for a, b in pairs], 1))
        q_orb = [min(q, n - 1 - q) for q in range(n)] if equivariant else list(range(n))
        mir = lambda p: frozenset((n - 1 - p[0], n - 1 - p[1])); key = [frozenset(p) for p in pairs]
        p_orb_raw = [min(i, key.index(mir(p))) for i, p in enumerate(pairs)] if equivariant else list(range(len(pairs)))
        uniq = {v: i for i, v in enumerate(sorted(set(p_orb_raw)))}; p_orb = [uniq[v] for v in p_orb_raw]
        s.q_orb, s.p_orb = q_orb, p_orb; nq, npo = max(q_orb) + 1, max(p_orb) + 1
        s.theta = nn.Parameter(0.3 * torch.randn(layers, nq, 3)); s.enc_w = nn.Parameter(torch.ones(layers, nq))
        s.gamma = nn.Parameter(0.3 * torch.randn(layers, npo)); s.head = nn.Linear(n + len(pairs), n_out)
        s.n_pairs = len(pairs)
    def features(s, x):
        B, n = x.shape[0], s.n
        if s.enc == 'amplitude': state = torch.complex(x, torch.zeros_like(x))
        else:
            state = torch.zeros(B, 2 ** n, dtype=torch.complex64); state[:, 0] = 1.0
        for l in range(s.L):
            if s.enc == 'angle' and (s.reup or l == 0):                                  # (re-)upload data
                for q in range(n): state = apply_1q(state, RY(s.enc_w[l, s.q_orb[q]] * x[:, q]), q, n)
            rot = RZ(s.theta[l, :, 0]) @ RY(s.theta[l, :, 1]) @ RZ(s.theta[l, :, 2])      # (nq, 2, 2) shared per mirror orbit
            for q in range(n): state = apply_1q(state, rot[s.q_orb[q]], q, n)
            g = s.gamma[l][s.p_orb]; phase = s.zz @ g                                      # diagonal parameterised ZZ entanglers
            state = state * torch.polar(torch.ones_like(phase), -0.5 * phase)
        probs = state.real ** 2 + state.imag ** 2
        return torch.cat([probs @ s.zs, probs @ s.zz], -1)                                # <Z_i>, <Z_i Z_j> measurements
    def forward(s, x): return s.head(s.features(x))
    def circuit_depth(s):                                                                  # analytic (CNOT-RZ-CNOT ZZ decomposition)
        colours = 2 if s.n % 2 == 0 else 3
        return s.L * ((1 if s.enc == 'angle' else 0) + 3 + 3 * colours) + (0 if s.enc == 'angle' else s.n)

@torch.no_grad()
def check_quantum_equivariance(model):
    x = torch.rand(32, model.n) * math.pi
    f1 = model.features(x)[:, :model.n]; f2 = model.features(x.flip(-1))[:, :model.n]
    return (f2 - f1.flip(-1)).abs().max().item()

class ClassicalMLP(nn.Module):
    def __init__(s, d_in, hidden=16, n_out=6):
        super().__init__(); s.net = nn.Sequential(nn.Linear(d_in, hidden), nn.GELU(), nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, n_out))
    def forward(s, x): return s.net(x)

count_params = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

def train_head_model(model, Xtr, Ytr, Xva, Yva, epochs, lr=0.03, bs=256, verbose=False, tag=''):
    opt = torch.optim.Adam(model.parameters(), lr=lr); best, best_state, hist = 1e9, None, []; t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr)); tl = []
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]; loss = mt_loss(model(Xtr[b]), Ytr[b]); opt.zero_grad(); loss.backward(); opt.step(); tl.append(loss.item())
        model.eval()
        with torch.no_grad(): vl = mt_loss(model(Xva), Yva).item()
        hist.append((float(np.mean(tl)), vl))
        if vl < best: best, best_state = vl, copy.deepcopy(model.state_dict())
        if verbose and (ep % 5 == 0 or ep == epochs - 1): print(f'  [{tag}] ep {ep:3d} train {np.mean(tl):.4f} val {vl:.4f}')
    model.load_state_dict(best_state); model.eval(); return dict(hist=hist, time=time.time() - t0)

def epochs_to_converge(hist, tol=0.05):
    v = np.array([h[1] for h in hist]); return int(np.argmax(v <= v.min() * (1 + tol)) + 1)

In [ ]:
# ----------------------------------------------------------------------------------------------------------
# Stages 3-6 as one experiment (used for the main run and for the ablation study)
# ----------------------------------------------------------------------------------------------------------
def run_experiment(D, TD, G, over=None, verbose=False, seed=0):
    c = dict(use_hyper=True, use_temporal=True, rewfe_equiv=True, q_kind='reqwn', encoding='angle', reupload=True, q_equiv=True,
             layers=CFG['Q_LAYERS'], n_qubits=CFG['N_QUBITS'], ep_graph=EP['graph'], ep_q=EP['q'], lr_q=0.03)
    c.update(over or {}); torch.manual_seed(seed); np.random.seed(seed)
    key = (c['use_hyper'], c['use_temporal'], c['rewfe_equiv'], c['ep_graph'], seed)
    if key not in S34_CACHE:
        s34 = Stage34(D['emp_X'].shape[1], D['task_X'].shape[1], D['world'].K, D['world'].D, CFG['D'], c['use_hyper'], c['use_temporal'], c['rewfe_equiv'])
        h34 = train_stage34(s34, G, TD, c['ep_graph'], verbose=verbose); Z, cache = extract_embeddings(s34, G, TD)
        S34_CACHE[key] = (s34, h34, Z, cache)
    s34, h34, Z, cache = S34_CACHE[key]; idx = TD['idx']
    mode = 'angle' if c['q_kind'] == 'mlp' else c['encoding']
    qpre = QPre(c['n_qubits'], mode).fit(Z[idx['train']].numpy()); X = qpre(Z)
    if c['q_kind'] == 'reqwn':
        q = REQWN(c['n_qubits'], c['layers'], 6, c['encoding'], c['reupload'], c['q_equiv'])
    else: q = ClassicalMLP(X.shape[1], hidden=8)
    tr = train_head_model(q, X[idx['train']], TD['y'][idx['train']], X[idx['val']], TD['y'][idx['val']], c['ep_q'], lr=c['lr_q'], verbose=verbose, tag='REQ-WN')
    with torch.no_grad(): lg = q(X)
    prob = torch.sigmoid(lg[:, :5]).numpy(); prod = (lg[:, 5] * TD['ps'] + TD['pm']).numpy()
    Yraw = D['Y']; df, reg = eval_multitask(prob, prod, Yraw, idx)
    return dict(cfg=c, s34=s34, h34=h34, Z=Z, cache=cache, qpre=qpre, X=X, q=q, trq=tr, prob=prob, prod=prod, table=df, reg=reg)

In [ ]:
print('Quantum circuit mirror-equivariance error:', check_quantum_equivariance(REQWN(CFG['N_QUBITS'], CFG['Q_LAYERS'])))
R = run_experiment(D, TD, G, verbose=True)
print('REQ-WN parameters:', count_params(R['q']), '| qubits:', CFG['N_QUBITS'], '| analytic circuit depth:', R['q'].circuit_depth())
print('Test-block metrics per target (REQ-WN):'); display(R['table'].round(3)); print('Productivity regression:', {k: round(v, 3) for k, v in R['reg'].items()})

## Stage 7 — Task Allocation and Workforce Control
**Constrained reinforcement learning → multi-agent policy learning → reward shaping → constraint-aware action selection.**  
One actor per department chooses an employee for each arriving task from REQ-WN predictions + skills + workload + priority + deadline + recent performance. Hard constraints mask infeasible employees (minimum skill match, maximum projected load); a Lagrange multiplier enforces a workload-risk budget; the shaped reward combines priority-weighted completion, timeliness, productivity and imbalance. Policies are trained on the validation-period stream and evaluated on the unseen test-period stream (common random numbers for all policies).

In [ ]:
# ----------------------------------------------------------------------------------------------------------
# End-to-end predictor: (employee, task, context) -> TERRN pair embedding -> RE-WFE -> PCA/encoding -> REQ-WN
# ----------------------------------------------------------------------------------------------------------
class HRPredictor:
    def __init__(s, s34, cache, TD, qpre, qmodel):
        s.s34, s.Ee, s.Te, s.TD, s.qpre, s.q = s34, cache[0], cache[1], TD, qpre, qmodel
    def clone(s):                                    # independent copy of the quantum model (used for online adaptation)
        c = copy.copy(s); c.q = copy.deepcopy(s.q); return c
    @torch.no_grad()
    def qfeatures(s, e, t, scal_raw, seq):
        e = torch.as_tensor(e).long(); t = torch.as_tensor(t).long(); scal_raw, seq = tt(scal_raw), tt(seq)
        scal = ((scal_raw - s.TD['mu']) / s.TD['sd']).clamp(-5, 5)
        z = s.s34.terrn.pair_embed(s.Ee, s.Te, e, t, scal, seq); zr = s.s34.rewfe(z)
        return s.q.features(s.qpre(zr))
    @torch.no_grad()
    def predict(s, e, t, scal_raw, seq):
        lg = s.q.head(s.qfeatures(e, t, scal_raw, seq))
        return torch.sigmoid(lg[:, :5]).numpy(), (lg[:, 5] * s.TD['ps'] + s.TD['pm']).numpy()   # P[complete,risk,delay,suit,imb], prod

In [ ]:
# ----------------------------------------------------------------------------------------------------------
# Environment (replays a task stream; common random numbers make policies comparable)
# ----------------------------------------------------------------------------------------------------------
def warm_history(state, D, start_time):
    log, w = D['log'], D['world']
    for r in log[log.arrival < start_time].itertuples():
        state.hist[r.emp_id].append((r.arrival + r.turn / w.cap[r.emp_id], r.delay, r.prod, r.quality, r.lr_before, math.log(w.est[r.task_id]) / 5))

class StreamEnv:
    def __init__(s, world, D, task_ids, seed, drift_at=None, drift_frac=0.3, drift_mag=0.35, L=5):
        rng = np.random.default_rng(seed); s.w, s.tasks = world, task_ids
        s.state = EmpState(world, L, init_load=rng.uniform(0, 0.9, world.E) * world.cap * 5); s.state.t = world.arrival[task_ids[0]]
        warm_history(s.state, D, world.arrival[task_ids[0]])
        s.drift = np.zeros(world.E); s.drift_at, s.hit, s.mag = drift_at, rng.random(world.E) < drift_frac, drift_mag
    def begin(s, k, t):
        if s.drift_at is not None and k == s.drift_at: s.drift[s.hit] = s.mag             # performance drift (e.g. burnout)
        now = s.w.arrival[t]; s.state.advance(now); return now
    def step(s, e, t, now):
        return s.state.assign(e, t, s.w.outcome(e, t, s.state.load[e], drift=s.drift[e]), now)

def cand_context(world, state, t, now, pred):
    c = world.dept_members[world.tdept[t]]; lr = state.load_ratio(c); seq, hd = state.seq(c, now)
    scal = world.scal(c, t, lr, hd); P, prod = pred.predict(c, np.full(len(c), t), scal, seq)
    lr_after = lr + world.est[t] / (world.cap[c] * 5); n = len(c)
    feats = np.column_stack([P, prod, scal[:, 0], lr, lr_after, scal[:, 1], hd, np.full(n, world.prio[t] / 3),
                             np.full(n, world.deadline[t] / world.est[t] / 3)]).astype(np.float32)
    return dict(t=t, cands=c, lr=lr, lr_after=lr_after, hd=hd, scal=scal, seq=seq, P=P, prod=prod, feats=feats)

def feasible_mask(ctx, min_skill=0.2, max_load=1.6):               # hard constraints (constraint-aware action selection)
    m = (ctx['scal'][:, 0] >= min_skill) & (ctx['lr_after'] <= max_load)
    return m if m.any() else ctx['lr_after'] <= ctx['lr_after'].min() + 1e-9

def shaped_reward(info, prio):                                      # reward shaping
    return (1 + 0.5 * (prio - 1)) * info['complete'] + 0.5 * (1 - info['delay']) + 0.3 * info['prod'] - 0.3 * info['imbalance']

class Agent(nn.Module):
    def __init__(s, d_in=26): super().__init__(); s.net = nn.Sequential(nn.Linear(d_in, 32), nn.GELU(), nn.Linear(32, 1))
    def forward(s, f): return s.net(torch.cat([f, f - f.mean(0, keepdim=True)], -1))[:, 0]

def agent_logits(agents, world, ctx):
    lg = agents[world.tdept[ctx['t']]](tt(ctx['feats'])); return lg.masked_fill(~torch.as_tensor(feasible_mask(ctx)), -1e9)

def train_constrained_rl(world, D, pred, tasks, episodes, risk_budget=0.10, lr=3e-3, update_every=32, verbose=True):
    """Multi-agent (one actor per department) REINFORCE with reward shaping and a Lagrangian workload-risk constraint."""
    agents = nn.ModuleList([Agent() for _ in range(world.D)]); opt = torch.optim.Adam(agents.parameters(), lr=lr)
    lam, base, curve = 0.0, 0.0, []
    for ep in range(episodes):
        env = StreamEnv(world, D, tasks, seed=1000 + ep); buf = []; R = []
        for k, t in enumerate(tasks):
            now = env.begin(k, t); ctx = cand_context(world, env.state, t, now, pred)
            dist = torch.distributions.Categorical(logits=agent_logits(agents, world, ctx)); a = dist.sample()
            info = env.step(ctx['cands'][a.item()], t, now); r = shaped_reward(info, world.prio[t]); R.append((r, info['risk'], info['delay']))
            buf.append((dist.log_prob(a), dist.entropy(), r, info['risk']))
            if len(buf) == update_every:
                lp = torch.stack([b[0] for b in buf]); en = torch.stack([b[1] for b in buf]); r_ = tt([b[2] for b in buf]); c_ = tt([b[3] for b in buf])
                adv = r_ - lam * c_ - base; loss = -(adv.detach() * lp).mean() - 0.01 * en.mean()
                opt.zero_grad(); loss.backward(); opt.step(); base = 0.95 * base + 0.05 * float((r_ - lam * c_).mean())
                lam = max(0.0, lam + 0.5 * (float(c_.mean()) - risk_budget)); buf = []
        R = np.array(R); curve.append(R.mean(0))
        if verbose: print(f'  [RL] episode {ep+1}/{episodes}  reward {R[:,0].mean():.3f}  risk {R[:,1].mean():.3f}  delay {R[:,2].mean():.3f}  lambda {lam:.2f}')
    return agents, np.array(curve)

In [ ]:
# ----------------------------------------------------------------------------------------------------------
# Stream runner + HR-management metrics
# ----------------------------------------------------------------------------------------------------------
def run_stream(world, D, pred, tasks, choose, seed, drift_at=None, observe=None, keep_ctx=False):
    env = StreamEnv(world, D, tasks, seed, drift_at=drift_at); rows, kept = [], []
    for k, t in enumerate(tasks):
        now = env.begin(k, t); ctx = cand_context(world, env.state, t, now, pred); ctx['k'] = k
        i = int(choose(ctx)); e = ctx['cands'][i]
        oc = world.outcome(ctx['cands'], t, env.state.load[ctx['cands']], drift=env.drift[ctx['cands']])   # oracle (evaluation only)
        sc = oc['quality'] - 0.5 * np.clip(oc['turn'] / world.deadline[t] - 1, 0, None)
        topk = np.argsort(-sc)[:max(1, math.ceil(0.1 * len(sc)))]
        info = env.step(e, t, now); info.update(k=k, t=t, e=e, match=float(i in topk), prio=world.prio[t], est=world.est[t],
                                                  lr_std=float(env.state.load_ratio().std()), p_delay=float(ctx['P'][i, 2]))
        rows.append(info)
        if observe is not None: observe(ctx, i, info)
        if keep_ctx: kept.append(dict(e=e, t=t, scal=ctx['scal'][i].copy(), seq=ctx['seq'][i].copy(), k=k))
    return pd.DataFrame(rows), kept

def stream_metrics(R):
    return dict(completion_rate=R.complete.mean(), delay_rate=R.delay.mean(), risk_rate=R.risk.mean(), productivity=R['prod'].mean(),
                workload_imbalance=R.lr_std.mean(), matching_accuracy=R.match.mean(),
                allocation_efficiency=(R.est * R.prio * R.complete).sum() / (R.est * R.prio).sum())

rng_pol = np.random.default_rng(7)
pol_random = lambda ctx: rng_pol.integers(len(ctx['cands']))
pol_least_loaded = lambda ctx: int(np.argmin(ctx['lr']))
pol_skill = lambda ctx: int(np.argmax(ctx['scal'][:, 0]))
def pol_pred_greedy(ctx):
    sc = ctx['P'][:, 0] + ctx['P'][:, 3] - ctx['P'][:, 2] - ctx['P'][:, 1] - ctx['P'][:, 4]
    return int(np.argmax(np.where(feasible_mask(ctx), sc, -1e9)))
def make_pol_rl(agents, world):
    return lambda ctx: int(torch.argmax(agent_logits(agents, world, ctx)).item())

In [ ]:
world = D['world']; pred = HRPredictor(R['s34'], R['cache'], TD, R['qpre'], R['q'])
val_tasks = D['log'].task_id.values[TD['idx']['val']][:EP['rl_tasks']]; test_tasks = D['log'].task_id.values[TD['idx']['test']][:EP['stream']]
agents, rl_curve = train_constrained_rl(world, D, pred, val_tasks, EP['rl_eps'])

In [ ]:
pols = {'Random': pol_random, 'Least-loaded': pol_least_loaded, 'Skill-greedy': pol_skill, 'REQNN-greedy': pol_pred_greedy,
        'Constrained-RL (ours)': make_pol_rl(agents, world)}
res7, runs7 = {}, {}
for name, p in pols.items():
    runs7[name], _ = run_stream(world, D, pred, test_tasks, p, seed=50); res7[name] = stream_metrics(runs7[name])
T7 = pd.DataFrame(res7).T; T7['delay_reduction_vs_random'] = 1 - T7['delay_rate'] / T7.loc['Random', 'delay_rate']
T7['productivity_gain_vs_random'] = T7['productivity'] / T7.loc['Random', 'productivity'] - 1
display(T7.round(3))

## Stage 8 — Dynamic HR Management
**Contextual bandits → online learning → temporal drift detection → adaptive policy updating.**  
Loop: *new task → employee assignment → task execution → observed performance → policy update*. A synthetic **performance drift** (e.g. burnout in 30 % of employees) is injected at 40 % of the test stream. Compared: (a) the static RL policy, (b) + contextual bandit (policy-prior LinUCB), (c) + online fine-tuning of the REQ-WN readout and Page-Hinkley drift detection that decays the bandit memory and boosts exploration.

In [ ]:
# ----------------------------------------------------------------------------------------------------------
# Stage 8 - Dynamic HR management: contextual bandit + online learning + drift detection + adaptive policy update
# ----------------------------------------------------------------------------------------------------------
class PageHinkley:
    def __init__(s, delta=0.03, lam=15.0): s.delta, s.lam = delta, lam; s.reset()
    def reset(s): s.n, s.mean, s.cum, s.mn = 0, 0.0, 0.0, 0.0
    def update(s, x):
        s.n += 1; s.mean += (x - s.mean) / s.n; s.cum += x - s.mean - s.delta; s.mn = min(s.mn, s.cum); return (s.cum - s.mn) > s.lam

class LinUCB:
    """Shared-parameter linear UCB with a prior mean that starts from the offline policy score (prior_dim)."""
    def __init__(s, d, alpha=0.3, lam=5.0, prior_dim=0):
        s.d, s.alpha, s.lam = d, alpha, lam; s.th0 = np.zeros(d); s.th0[prior_dim] = 1.0
        s.A = lam * np.eye(d); s.b = lam * s.th0
    def scores(s, X, boost=1.0):
        Ai = np.linalg.inv(s.A); th = Ai @ s.b
        return X @ th + boost * s.alpha * np.sqrt(np.einsum('nd,dk,nk->n', X, Ai, X))
    def update(s, x, r): s.A += np.outer(x, x); s.b += r * x
    def decay(s, g): s.A = g * s.A + (1 - g) * s.lam * np.eye(s.d); s.b = g * s.b + (1 - g) * s.lam * s.th0

class AdaptiveManager:
    def __init__(s, agents, world, pred, TD, online=True, detect=True, bandit=True):
        s.agents, s.w, s.pred, s.TD = agents, world, pred.clone(), TD; s.online, s.detect, s.bandit = online, detect, bandit
        s.ucb = LinUCB(7); s.ph = PageHinkley(); s.resid = np.zeros(world.E); s.boost_left = 0; s.alarms = []
        s.buf_f, s.buf_y = [], []; s.opt = torch.optim.Adam(s.pred.q.head.parameters(), lr=0.01); s.nsteps = 0
    def _X(s, ctx, lgv):
        lgc = lgv - lgv.mean()
        return np.column_stack([lgc, ctx['P'][:, 2], ctx['P'][:, 0], ctx['lr_after'], ctx['hd'], 2 * s.resid[ctx['cands']], np.ones(len(lgc))])
    def choose(s, ctx):
        lg = agent_logits(s.agents, s.w, ctx).detach().numpy(); mask = feasible_mask(ctx)
        if not s.bandit: return int(np.argmax(lg))
        X = s._X(ctx, np.where(mask, lg, lg[mask].mean()))
        sc = s.ucb.scores(X, boost=2.0 if s.boost_left > 0 else 1.0); s.boost_left = max(0, s.boost_left - 1)
        s._lastX = X; return int(np.argmax(np.where(mask, sc, -1e9)))
    def observe(s, ctx, i, info):
        e = ctx['cands'][i]; err = info['delay'] - ctx['P'][i, 2]; s.resid[e] = 0.7 * s.resid[e] + 0.3 * err
        if s.bandit: s.ucb.update(s._lastX[i], shaped_reward(info, s.w.prio[ctx['t']]) - info['risk'])
        if s.detect and s.ph.update(err):                                  # temporal drift detected -> adapt
            s.alarms.append(ctx['k']); s.ph.reset(); s.ucb.decay(0.3); s.boost_left = 60; s.buf_f, s.buf_y = s.buf_f[-60:], s.buf_y[-60:]
        if s.online:                                                       # online learning of the REQ-WN readout head
            f = s.pred.qfeatures([e], [ctx['t']], ctx['scal'][i:i + 1], ctx['seq'][i:i + 1])
            y = np.array([info['complete'], info['risk'], info['delay'], info['suit'], info['imbalance'], (info['prod'] - s.TD['pm']) / s.TD['ps']], np.float32)
            s.buf_f.append(f[0]); s.buf_y.append(tt(y)); s.buf_f, s.buf_y = s.buf_f[-200:], s.buf_y[-200:]; s.nsteps += 1
            if s.nsteps % 20 == 0 and len(s.buf_f) >= 40:
                Fb, Yb = torch.stack(s.buf_f), torch.stack(s.buf_y); s.pred.q.head.train()
                for _ in range(3): loss = mt_loss(s.pred.q.head(Fb), Yb); s.opt.zero_grad(); loss.backward(); s.opt.step()

In [ ]:
drift_at = int(0.4 * len(test_tasks)); runs8 = {}
runs8['Static RL'], _ = run_stream(world, D, pred, test_tasks, make_pol_rl(agents, world), seed=50, drift_at=drift_at)
mgr_b = AdaptiveManager(agents, world, pred, TD, online=False, detect=False, bandit=True)
runs8['+ Contextual bandit'], _ = run_stream(world, D, mgr_b.pred, test_tasks, mgr_b.choose, seed=50, drift_at=drift_at, observe=mgr_b.observe)
mgr_f = AdaptiveManager(agents, world, pred, TD, online=True, detect=True, bandit=True)
runs8['+ Online learning + drift detection'], kept = run_stream(world, D, mgr_f.pred, test_tasks, mgr_f.choose, seed=50, drift_at=drift_at, observe=mgr_f.observe, keep_ctx=True)
print('Page-Hinkley drift alarms at stream steps:', mgr_f.alarms, '(true drift at', drift_at, ')')
rows = {}
for n, r in runs8.items():
    a, b = r[r.k < drift_at], r[r.k >= drift_at]
    rows[n] = dict(delay_pre=a.delay.mean(), delay_post=b.delay.mean(), completion_pre=a.complete.mean(), completion_post=b.complete.mean(), risk_post=b.risk.mean(), productivity_post=b['prod'].mean())
T8 = pd.DataFrame(rows).T; display(T8.round(3))
plt.figure(figsize=(7, 3))
for n, r in runs8.items(): plt.plot(r.delay.rolling(60, min_periods=20).mean().values, label=n)
plt.axvline(drift_at, c='k', ls='--', label='drift'); plt.ylabel('rolling delay rate'); plt.xlabel('task #'); plt.legend(fontsize=7); plt.tight_layout(); plt.show()

## Stage 9 — Explainable HR Decision Support
**SHAP → counterfactual explanations → feature attribution → what-if analysis.**  
Shapley values are estimated by permutation sampling (the model-agnostic estimator behind SHAP; efficiency property holds exactly) over the interpretable pair-level inputs of the *full REQNN pipeline*, with the employee's temporal history held fixed. What-if analysis re-runs the pipeline with modified inputs (e.g. lower workload).

In [ ]:
# ----------------------------------------------------------------------------------------------------------
# Stage 9 - Explainability: Shapley values (permutation sampling) + counterfactual / what-if on the full REQNN pipeline
# ----------------------------------------------------------------------------------------------------------
def shapley_sampling(f, x, base, n_perm=100, seed=0):
    """f: (m, F) -> (m,) ; returns Shapley values of f(x) - f(base) (efficiency property holds exactly)."""
    rng = np.random.default_rng(seed); Fn = len(x); rows, orders = [], []
    for _ in range(n_perm):
        order = rng.permutation(Fn); cur = base.copy(); rows.append(cur.copy())
        for j in order: cur[j] = x[j]; rows.append(cur.copy())
        orders.append(order)
    vals = f(np.array(rows)).reshape(n_perm, Fn + 1); phi = np.zeros(Fn)
    for p, order in enumerate(orders):
        for k, j in enumerate(order): phi[j] += vals[p, k + 1] - vals[p, k]
    return phi / n_perm, float(vals[:, 0].mean()), float(vals[:, -1].mean())

def make_pair_fn(pred, e, t, seq, out_idx):
    def f(rows):
        n = len(rows); P, prod = pred.predict(np.full(n, e), np.full(n, t), rows.astype(np.float32), np.repeat(seq[None], n, 0))
        return P[:, out_idx] if out_idx < 5 else prod
    return f

def explain_assignment(pred, world, rec, base_scal, target='delay', n_perm=100):
    j = TARGETS.index(target); f = make_pair_fn(pred, rec['e'], rec['t'], rec['seq'], j)
    phi, fb, fx = shapley_sampling(f, rec['scal'].astype(np.float64), base_scal.astype(np.float64), n_perm)
    return pd.Series(phi, index=PAIR_FEATS).sort_values(key=np.abs, ascending=False), fb, fx

def workload_whatif(pred, rec, reductions=np.linspace(0, 0.6, 13), target='delay'):
    j = TARGETS.index(target); f = make_pair_fn(pred, rec['e'], rec['t'], rec['seq'], j); rows = np.repeat(rec['scal'][None].astype(np.float64), len(reductions), 0)
    rows[:, PAIR_FEATS.index('workload_ratio')] *= (1 - reductions); return f(rows)

In [ ]:
pred_x = mgr_f.pred; base_scal = D['scal'][TD['idx']['train']].mean(0); R8 = runs8['+ Online learning + drift detection']
post = R8[R8.k >= drift_at].sort_values('p_delay'); picks = [post.index[-1], post.index[len(post) // 2], post.index[0]]
fig, axs = plt.subplots(1, 3, figsize=(13, 3.2)); sweeps = []
for ax, pi_, tag in zip(axs, picks, ['highest', 'median', 'lowest']):
    rec = kept[pi_]
    phi, fb, fx = explain_assignment(pred_x, world, rec, base_scal, 'delay', n_perm=60 if FAST else 200)
    vals = dict(zip(PAIR_FEATS, rec['scal']))
    print(f"\nEmployee {rec['e']} assigned to Task {rec['t']}  ->  predicted delay probability {fx:.2f} (average assignment {fb:.2f})  [{tag} predicted risk]")
    for f, v in phi.head(4).items(): print(f"   {f} = {vals[f]:.2f}: {'raises' if v > 0 else 'lowers'} delay risk by {abs(v):.3f}")
    red = np.linspace(0, 0.6, 13); sw = workload_whatif(pred_x, rec, red, 'delay'); sweeps.append((red, sw))
    hit = [r for r, p in zip(red, sw) if p <= 0.8 * sw[0]]
    print(f"   What-if: workload -20% -> delay probability {sw[4]:.2f} (from {sw[0]:.2f}); " + (f'a {hit[0]:.0%} reduction cuts it by 20% relative' if hit else 'no tested workload reduction cuts it by 20% relative'))
    ax.barh(phi.index[::-1], phi.values[::-1]); ax.set_title(f'Shapley: delay prob ({tag})', fontsize=9); ax.axvline(0, c='k', lw=.5)
plt.tight_layout(); plt.show()
plt.figure(figsize=(4.5, 3))
for (red, sw), tag in zip(sweeps, ['highest', 'median', 'lowest']): plt.plot(red * 100, sw, label=tag)
plt.xlabel('workload reduction (%)'); plt.ylabel('predicted delay probability'); plt.legend(); plt.tight_layout(); plt.show()

## Stage 10 — Evaluation
**Prediction metrics:** Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, MCC.  **HR metrics:** completion rate, delay reduction, workload imbalance, matching accuracy, productivity improvement, allocation efficiency.  **Quantum/model metrics:** qubits, circuit depth, parameters, convergence, inference time, classical-vs-QNN comparison.  **Validation:** temporal hold-out (main protocol above), baseline comparison, ablation study, nested stratified CV.

In [ ]:
# ---- Baseline comparison on the temporal test block (same thresholds protocol) ----
idx = TD['idx']; summ, base_time = {}, {}
summ['REQ-WN (proposed)'] = summarize(R['table'], R['reg'])
with torch.no_grad(): lg = R['s34'].head(R['Z'])
tb, rg = eval_multitask(torch.sigmoid(lg[:, :5]).numpy(), (lg[:, 5] * TD['ps'] + TD['pm']).numpy(), D['Y'], idx); summ['TERRN+RE-WFE (classical head)'] = summarize(tb, rg)
for name in ['LogReg', 'RandForest', 'HistGB', 'MLP']:
    P, pr, tm = baseline_predictions(name, D['flat'], D['Y'], idx); t_, r_ = eval_multitask(P, pr, D['Y'], idx); summ[name + ' (tabular)'] = summarize(t_, r_); base_time[name] = tm
T10a = pd.DataFrame(summ).T; print('Mean over the 5 binary targets (+ productivity regression):'); display(T10a)

In [ ]:
# ---- HR-management metrics (Stages 7-8 on the unseen test stream) ----
hr = {k: v for k, v in res7.items()}
for n in ['Static RL', '+ Contextual bandit', '+ Online learning + drift detection']: hr[f'{n} [with drift]'] = stream_metrics(runs8[n])
T10b = pd.DataFrame(hr).T; rnd = T7.loc['Random']
T10b['delay_reduction_vs_random'] = 1 - T10b['delay_rate'] / rnd['delay_rate']; T10b['productivity_improvement'] = T10b['productivity'] / rnd['productivity'] - 1
display(T10b.round(3))

In [ ]:
# ---- Quantum / model metrics + classical-vs-QNN computational comparison ----
R_mlp = run_experiment(D, TD, G, dict(q_kind='mlp'))
def infer_ms(m, X):
    t0 = time.time()
    with torch.no_grad(): m(X[:1000])
    return 1000 * (time.time() - t0)
cmp = pd.DataFrame({
  'REQ-WN (simulated)': dict(qubits=CFG['N_QUBITS'], circuit_depth=R['q'].circuit_depth(), parameters=count_params(R['q']), epochs_to_converge=epochs_to_converge(R['trq']['hist']),
       train_s=R['trq']['time'], infer_ms_per_1000=infer_ms(R['q'], R['X']), statevector_bytes=2 ** CFG['N_QUBITS'] * 8),
  'Classical MLP head (same PCA input)': dict(qubits=0, circuit_depth=0, parameters=count_params(R_mlp['q']), epochs_to_converge=epochs_to_converge(R_mlp['trq']['hist']),
       train_s=R_mlp['trq']['time'], infer_ms_per_1000=infer_ms(R_mlp['q'], R_mlp['X']), statevector_bytes=0)}).T
display(cmp.round(3)); print('Tabular baselines training time (s):', {k: round(v, 1) for k, v in base_time.items()})
print('Note: REQ-WN time is classical simulation cost (grows as 2^qubits); circuit depth assumes CNOT-RZ-CNOT ZZ gates; on hardware inference cost is shots-limited.')
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(np.array(h34)); ax[0].set_title('TERRN+RE-WFE (train, val loss)'); ax[0].legend(['train', 'val'])
ax[1].plot(np.array(R['trq']['hist'])); ax[1].set_title('REQ-WN (train, val loss)'); ax[1].legend(['train', 'val']); plt.tight_layout(); plt.show()

In [ ]:
# ---- Ablation study (temporal test block; mean over targets) ----
abl = {'Full pipeline': {}, 'w/o hypergraph learning': dict(use_hyper=False), 'w/o temporal message passing': dict(use_temporal=False),
       'w/o reflection-equivariant encoder (plain MLP)': dict(rewfe_equiv=False), 'w/o circuit equivariance (untied parameters)': dict(q_equiv=False),
       'w/o data re-uploading': dict(reupload=False), 'amplitude instead of angle encoding': dict(encoding='amplitude'),
       'classical MLP instead of quantum circuit': dict(q_kind='mlp')}
rows = {}
for name, over in abl.items():
    r = R if not over else run_experiment(D, TD, G, over); rows[name] = summarize(r['table'], r['reg']); print('done:', name)
T10c = pd.DataFrame(rows).T; display(T10c)

In [ ]:
# ---- Nested stratified cross-validation on the training period (delay target, ROC-AUC) ----
tri = torch.as_tensor(TD['idx']['train']); Xn, Yn = R['X'][tri], TD['y'][tri]; strat = D['Y'][TD['idx']['train'], 2].astype(int)
def fit_eval(make, hp, Xa, Ya, Xb, Yb, epochs):
    perm = torch.randperm(len(Xa)); nv = int(0.15 * len(Xa)); va, tr = perm[:nv], perm[nv:]
    m = make(hp); train_head_model(m, Xa[tr], Ya[tr], Xa[va], Ya[va], epochs, lr=hp.get('lr', 0.03))
    with torch.no_grad(): p = torch.sigmoid(m(Xb)[:, 2]).numpy()
    return roc_auc_score(Yb[:, 2].numpy(), p)
def nested_cv(make, grid, X, Y, strat, epochs, outer=3, inner=2, seed=0):
    rows = []
    for fo, (tr, te) in enumerate(StratifiedKFold(outer, shuffle=True, random_state=seed).split(np.zeros(len(strat)), strat)):
        sc = []
        for hp in grid:
            s = [fit_eval(make, hp, X[tr][ti], Y[tr][ti], X[tr][vi], Y[tr][vi], epochs) for ti, vi in
                 StratifiedKFold(inner, shuffle=True, random_state=seed).split(np.zeros(len(tr)), strat[tr])]
            sc.append(np.mean(s))
        best = grid[int(np.argmax(sc))]; rows.append(dict(outer_fold=fo, best_hp=str(best), inner_AUC=max(sc), outer_AUC=fit_eval(make, best, X[tr], Y[tr], X[te], Y[te], epochs)))
    return pd.DataFrame(rows)
cv_q = nested_cv(lambda hp: REQWN(CFG['N_QUBITS'], hp['layers'], 6), [dict(layers=2, lr=0.03), dict(layers=3, lr=0.03)], Xn, Yn, strat, EP['cv'])
cv_c = nested_cv(lambda hp: ClassicalMLP(Xn.shape[1], hp['hidden']), [dict(hidden=8, lr=0.01), dict(hidden=16, lr=0.01)], Xn, Yn, strat, EP['cv'])
display(cv_q); display(cv_c); print(f"Nested-CV outer ROC-AUC (delay): REQ-WN {cv_q.outer_AUC.mean():.3f} ± {cv_q.outer_AUC.std():.3f} | classical MLP {cv_c.outer_AUC.mean():.3f} ± {cv_c.outer_AUC.std():.3f}")
print('Caveat: embeddings were fitted on training-period labels, so CV scores are optimistic; the temporal hold-out above is the clean estimate.')

In [ ]:
# ---- Save results ----
os.makedirs('outputs', exist_ok=True)
T10a.to_csv('outputs/prediction_summary.csv'); T10b.to_csv('outputs/hr_metrics.csv'); T10c.to_csv('outputs/ablation.csv'); cmp.to_csv('outputs/quantum_model_metrics.csv')
R['table'].to_csv('outputs/reqwn_per_target.csv'); torch.save({'stage34': R['s34'].state_dict(), 'reqwn': R['q'].state_dict()}, 'outputs/models.pt'); print('saved to ./outputs')

### Notes on extending this notebook
* **Real data:** build `emp` / `tasks` / assignment-log tables with the same columns (see `run_stage1`) and replace `World.outcome` with an outcome model fitted on your logs (or use the REQ-WN predictions themselves as the environment for offline RL).
* **Quantum hardware / libraries:** `REQWN.features` is a plain gate sequence (RY encoding, RZ·RY·RZ rotations, ZZ phase ring); it ports one-to-one to PennyLane/Qiskit (`RY`, `Rot`, `IsingZZ`). Use `FAST=False` and several seeds for final reporting.
* **Scale:** dense attention is fine up to a few thousand tasks; for larger graphs switch to sparse neighbour sampling.